# Q8 : la fenêtre SQL et l'objet groupby

In [12]:
import warnings
warnings.filterwarnings("ignore", message=".*SQLAlchemy.*")
import pandas as pd
import psycopg2

conn = psycopg2.connect("dbname=vetprice")

## Explication de WINDOW

Le jeu de données utilisé pour l'exemple : 

```
|    | cle   |   jour |   price |
|---:|:------|-------:|--------:|
|  0 | A     |      1 |      10 |
|  1 | A     |      2 |      12 |
|  2 | A     |      3 |       9 |
|  3 | B     |      1 |       5 |
|  4 | B     |      2 |       7 |
```


In [13]:
JOUET = """
WITH prix(cle, jour, price) AS (
    VALUES ('A', 1, 10.0), ('A', 2, 12.0), ('A', 3, 9.0),
           ('B', 1, 5.0),  ('B', 2, 7.0)
)
"""

print(pd.read_sql(JOUET + "SELECT * FROM prix", conn).to_markdown())

|    | cle   |   jour |   price |
|---:|:------|-------:|--------:|
|  0 | A     |      1 |      10 |
|  1 | A     |      2 |      12 |
|  2 | A     |      3 |       9 |
|  3 | B     |      1 |       5 |
|  4 | B     |      2 |       7 |


### 1. PARTITION BY permet de découper en groupe en gardant le même nombre de lignes

- chaque ligne reste
- on crée une colonne avec une valeur calculée par groupe

**La différence avec `GROUP BY` est la conservation des lignes**

In [14]:
pd.read_sql(JOUET + """
    SELECT cle, jour, price,
           COUNT(*) OVER (PARTITION BY cle) AS n_dans_groupe
    FROM prix
""", conn)

,cle,jour,price,n_dans_groupe
0,A,1,10.0,3
1,A,2,12.0,3
2,A,3,9.0,3
3,B,1,5.0,2
4,B,2,7.0,2


### 2. Avec ORDER BY, le cadre par défaut s'arrête à la ligne courante

- `LAST_VALUE` renvoie alors la ligne courante, pas le dernier prix du groupe. 
- **Hors, on veut le dernier prix du groupe**.

```
groupe A, trié par jour.
Cadre par défaut = du premier au COURANT.

  courant = jour 1   [10]           LAST_VALUE = 10
  courant = jour 2   [10 12]        LAST_VALUE = 12
  courant = jour 3   [10 12 9]      LAST_VALUE = 9

Le bord droit suit la ligne courante : "dernier" = prix courant,
pas le dernier prix du groupe.
```

In [16]:
# premier est la première valeur du groupe
# dernier_faux n'est pas toujours la dernière valeur du groupe
pd.read_sql(JOUET + """
    SELECT cle, jour, price,
           FIRST_VALUE(price) OVER (PARTITION BY cle ORDER BY jour) AS premier,
           LAST_VALUE(price)  OVER (PARTITION BY cle ORDER BY jour) AS dernier_faux
    FROM prix
""", conn)

,cle,jour,price,premier,dernier_faux
0,A,1,10.0,10.0,10.0
1,A,2,12.0,10.0,12.0
2,A,3,9.0,10.0,9.0
3,B,1,5.0,5.0,5.0
4,B,2,7.0,5.0,7.0


### 3. Cadre élargi

- `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` élargi le cadre à tout le groupe. 
- `LAST_VALUE` renvoie le vrai dernier prix.
- La clause `WINDOW` nomme le cadre une fois (qui est réutilisé par les deux fonctions `FIRST_VALUE` et `LAST_VALUE`). 

In [ ]:
pd.read_sql(JOUET + """
    SELECT cle, jour, price,
           FIRST_VALUE(price) OVER w AS premier,
           LAST_VALUE(price)  OVER w AS dernier
    FROM prix
    WINDOW w AS (
        PARTITION BY cle ORDER BY jour
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    )
""", conn)

## pandas : l'objet groupby


In [17]:
df = pd.DataFrame({
    "cle":   ["A", "A", "A", "B", "B"],
    "jour":  [1, 2, 3, 1, 2],
    "price": [10.0, 12.0, 9.0, 5.0, 7.0],
})
df

,cle,jour,price
0,A,1,10.0
1,A,2,12.0
2,A,3,9.0
3,B,1,5.0
4,B,2,7.0


### `g` ne calcule rien

On calcule un découpage en groupes, mais sans faire d'opération d'aggrégation

In [18]:
g = df.groupby("cle")["price"]
g

### `agg(list)` montre le contenu de chaque groupe

La liste des prix calculée groupe par groupe.
(l'objet renvoyé est une `pd.Series`)

In [26]:
g.agg(list)

cle
A    [10.0, 12.0, 9.0]
B           [5.0, 7.0]
Name: price, dtype: object

In [27]:
type(g.agg(list))

pandas.core.series.Series

### Une fonction stat calcule une valeur par groupe
(l'objet renvoyé est une `pd.Series`)

In [28]:
g.mean()

cle
A    10.333333
B     6.000000
Name: price, dtype: float64

### `first` et `last` : premier et dernier de chaque groupe

Dans l'ordre courant du dataframe (trier avant si besoin). Pendant pandas de `FIRST_VALUE` / `LAST_VALUE`.

In [29]:
pd.DataFrame({"premier": g.first(), "dernier": g.last()})

,premier,dernier
cle,,
A,10.0,9.0
B,5.0,7.0
